# Ridge Baseline on GPU — Colab Master Notebook

End-to-end execution of the `data-ridge-baseline` pipeline with a vectorized alpha sweep on Colab T4 / L4 / A100 GPUs. Sweeping **thousands of L2 regularization values finishes in seconds** because we eigendecompose `X.T @ X` exactly once and reduce every additional alpha to an elementwise broadcast.

**Pipeline**
1. Environment setup + dependency install
2. Repo access (clone from GitHub *or* mount Drive)
3. Preprocessing pipeline (or load cached `.npz`)
4. Vectorized ridge sweep across event budgets × ~1000 alphas
5. Save canonical JSON results + secondary efficiency metrics
6. Quick visualization of the accuracy–efficiency frontier

Project: **EE207 Neuromorphic BCI** · branch `data-ridge-baseline`

## 1. Environment

Most of these are already on Colab. The `--quiet --upgrade` is a no-op if the right version is already installed.

In [ ]:
%pip install --quiet --upgrade \
    numpy scipy scikit-learn pandas matplotlib pyyaml h5py tqdm \
    nlb-tools dandi

In [ ]:
import platform, sys
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"python    : {sys.version.split()[0]}")
print(f"platform  : {platform.platform()}")
print(f"torch     : {torch.__version__}")
print(f"device    : {device}")
if device == "cuda":
    print(f"gpu       : {torch.cuda.get_device_name(0)}")
    print(f"cuda      : {torch.version.cuda}")

## 2. Repo access

Two options:
* **Clone from GitHub** (default) — pulls the `data-ridge-baseline` branch fresh into `/content`.
* **Mount Drive** — set `USE_DRIVE = True` and point `REPO_DIR` at your synced copy.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL  = "https://github.com/manrajmondair/neuromorphic-bci.git"
BRANCH    = "data-ridge-baseline"
REPO_DIR  = Path("/content/neuromorphic-bci")
USE_DRIVE = False  # flip to True to mount Drive instead of cloning

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/drive/MyDrive/neuromorphic-bci")
elif not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
print(f"working dir : {os.getcwd()}")
print(f"branch      : {head}")

In [ ]:
import logging
import numpy as np
import torch
from pathlib import Path

from src.data.preprocess import preprocess_mc_rtt, save_processed, load_processed
from src.features.event_budget import restrict_to_event_budget
from src.evaluation.metrics import velocity_r2
from src.evaluation.experiment_runner import save_json_results
from src.evaluation.efficiency_tracker import (
    compute_efficiency_summary, save_efficiency_json,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
print("imports OK")

## 3. Preprocessing

If `data/processed/processed_mc_rtt.npz` already exists we just load it; otherwise we download the NLB MC_RTT dandiset and run the full preprocessing pipeline. Persist the .npz to Drive (when mounted) for instant re-runs.

In [ ]:
raw_dir        = Path("data/raw")
processed_path = Path("data/processed/processed_mc_rtt.npz")

# Pull the dandiset if absent
if not any(raw_dir.rglob("*.nwb")):
    print("downloading MC_RTT from DANDI ...")
    subprocess.run([sys.executable, "scripts/download_mc_rtt.py"], check=True)

if processed_path.is_file():
    print(f"loading cached {processed_path}")
    data = load_processed(processed_path)
else:
    print("running preprocessing pipeline ...")
    data = preprocess_mc_rtt(raw_dir=raw_dir, bin_size_ms=50)
    save_processed(data, processed_path)

num_bins, num_neurons = data["spike_counts"].shape
print(f"num_bins={num_bins}  num_neurons={num_neurons}")
print(
    f"train={data['train_idx'].size}  "
    f"val={data['val_idx'].size}  "
    f"test={data['test_idx'].size}"
)

## 4. Vectorized ridge sweep on GPU

Closed-form ridge with broadcasting over alpha. For a training matrix `X ∈ R^(n×N)` and targets `y ∈ R^(n×K)`:

$$
W_\alpha = (X^\top X + \alpha I)^{-1} X^\top y
$$

Factor `X^T X = V \mathrm{diag}(\lambda) V^\top` once. Then for every candidate `α` the inverse is just `V \mathrm{diag}((\lambda + \alpha)^{-1}) V^\top`. With `z = V^\top X^\top y` and `u = X_\text{eval} V`, predictions for the entire alpha grid become

$$
\hat y_\alpha = u \cdot \frac{z}{\lambda + \alpha}.
$$

That's `O(N^3)` once plus `O(A \cdot m \cdot N)` for `A` alphas — milliseconds for `A \approx 1000` on T4.

In [ ]:
def batch_ridge_predict(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_eval:  torch.Tensor,
    alphas:  torch.Tensor,
) -> torch.Tensor:
    """Closed-form ridge for many alphas in a single GPU pass.

    X_train : [n, N]      training features (float64 recommended)
    y_train : [n, K]      training targets
    X_eval  : [m, N]      features to predict on
    alphas  : [A]         L2 regularization grid

    Returns : [A, m, K]   prediction tensor across all alphas.
    """
    XtX = X_train.T @ X_train                        # [N, N]
    Xty = X_train.T @ y_train                        # [N, K]
    evals, evecs = torch.linalg.eigh(XtX)            # symmetric PSD
    z = evecs.T @ Xty                                # [N, K]
    u = X_eval @ evecs                               # [m, N]
    denom = evals.unsqueeze(0) + alphas.unsqueeze(1)  # [A, N]
    filtered = z.unsqueeze(0) / denom.unsqueeze(-1)   # [A, N, K]
    return torch.einsum("mn,ank->amk", u, filtered)   # [A, m, K]


def joint_r2_per_alpha(y_true: torch.Tensor, preds: torch.Tensor) -> torch.Tensor:
    """Joint R^2 across both velocity axes, vectorized over alphas."""
    mean_v = y_true.mean(dim=0, keepdim=True)
    ss_tot = ((y_true - mean_v) ** 2).sum()
    ss_res = ((preds - y_true.unsqueeze(0)) ** 2).sum(dim=(1, 2))  # [A]
    return 1.0 - ss_res / ss_tot

### 4b. Executive loop — event budgets × alphas × seeds

For every `(seed, event_budget)`:
1. Filter events to the earliest fraction `f` and rebuild dense spike_counts.
2. Move features to GPU once.
3. Sweep all `N_ALPHAS` regularization values on the validation split.
4. Refit the best `α` and score on the held-out test split.
5. Accumulate canonical-schema row.

In [ ]:
EVENT_BUDGETS = (1.00, 0.50, 0.25, 0.10)
SEEDS         = (0, 1, 2)
N_ALPHAS      = 1000          # dense log-spaced grid
ALPHA_LO_HI   = (1e-4, 1e5)
DTYPE         = torch.float64  # eigendecomp is well-conditioned in float64

alphas    = torch.logspace(
    np.log10(ALPHA_LO_HI[0]), np.log10(ALPHA_LO_HI[1]),
    N_ALPHAS, device=device, dtype=DTYPE,
)
y         = torch.tensor(np.asarray(data["velocity"], dtype=np.float64), device=device, dtype=DTYPE)
train_idx = torch.tensor(data["train_idx"], device=device)
val_idx   = torch.tensor(data["val_idx"], device=device)
test_idx  = torch.tensor(data["test_idx"], device=device)
n_events_total = int(sum(t.size for t in data["event_times"]))

rows = []
import time
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    for f in EVENT_BUDGETS:
        sub = restrict_to_event_budget(data, fraction=f)
        X = torch.tensor(sub["spike_counts"].astype(np.float64), device=device, dtype=DTYPE)

        t0 = time.time()
        preds_val = batch_ridge_predict(X[train_idx], y[train_idx], X[val_idx], alphas)
        r2_val    = joint_r2_per_alpha(y[val_idx], preds_val)
        best_a    = int(torch.argmax(r2_val).item())
        best_alpha = float(alphas[best_a].item())
        # Score best on test
        preds_test = batch_ridge_predict(
            X[train_idx], y[train_idx], X[test_idx], alphas[best_a:best_a+1],
        )[0]
        elapsed = time.time() - t0

        y_test = y[test_idx]
        ss_res_ax = ((preds_test - y_test) ** 2).sum(dim=0).cpu().numpy()
        ss_tot_ax = ((y_test - y_test.mean(0, keepdim=True)) ** 2).sum(dim=0).cpu().numpy()
        r2_ax     = 1.0 - ss_res_ax / np.where(ss_tot_ax == 0, 1.0, ss_tot_ax)
        r2_joint  = 1.0 - ss_res_ax.sum() / ss_tot_ax.sum()

        n_events_used = int(sum(t.size for t in sub["event_times"]))
        rows.append({
            "model":          "ridge",
            "event_budget":   float(f),
            "seed":           int(seed),
            "r2_vx":          float(r2_ax[0]),
            "r2_vy":          float(r2_ax[1]),
            "r2_joint":       float(r2_joint),
            "best_alpha":     best_alpha,
            "n_events_used":  n_events_used,
            "n_events_total": n_events_total,
            "notes":          f"gpu sweep over {N_ALPHAS} alphas in {elapsed:.2f}s",
        })
        print(
            f"seed={seed} f={f:.2f}  best α={best_alpha:.4g}  "
            f"r2_joint={r2_joint:+.4f}  vx={r2_ax[0]:+.4f}  vy={r2_ax[1]:+.4f}  "
            f"({elapsed:.2f}s for {N_ALPHAS} alphas)"
        )

## 5. Save canonical results + efficiency summary

In [ ]:
results_dir = Path("results/ridge")
results_dir.mkdir(parents=True, exist_ok=True)

config = {
    "processed_path":   str(processed_path),
    "bin_size_ms":      int(data["bin_size_ms"]),
    "num_neurons":      int(data["num_neurons"]),
    "event_budgets":    list(EVENT_BUDGETS),
    "seeds":            list(SEEDS),
    "n_alphas":         int(N_ALPHAS),
    "alpha_lo_hi":      list(ALPHA_LO_HI),
    "device":           device,
    "dtype":            str(DTYPE),
    "notebook":         "02_ridge_baseline_colab.ipynb",
}
save_json_results(
    results_dir / "ridge_results.json",
    model="ridge",
    config=config,
    rows=rows,
)
save_efficiency_json(
    compute_efficiency_summary(data, fractions=EVENT_BUDGETS),
    results_dir / "computational_efficiency.json",
)
print("results saved")

## 6. Quick frontier visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df  = pd.DataFrame(rows)
agg = df.groupby("event_budget")["r2_joint"].agg(["mean", "std"]).reset_index()
agg = agg.sort_values("event_budget", ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    agg["event_budget"], agg["mean"], yerr=agg["std"],
    marker="o", linewidth=2, capsize=4, label="Ridge (GPU sweep)",
)
ax.axhline(0.0, color="black", linewidth=0.5, alpha=0.4)
ax.set_xlim(1.05, -0.02)
ax.set_ylim(-0.05, 1.0)
ax.set_xlabel("Event budget f")
ax.set_ylabel("Velocity R² (joint)")
ax.set_title("Ridge baseline on Colab — accuracy vs. event budget")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

### Next steps

* Persist `results/ridge/ridge_results.json` and `computational_efficiency.json` back to GitHub or Drive.
* Once the partner's SNN run lands at `results/snn/snn_results.json` and the shuffle control at `results/controls/shuffle_results.json`, run `python scripts/generate_final_figures.py` locally to render the headline frontier + qualitative trajectory figures at dpi=300.